### Computing committor for a random walk in maze

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math 
from mpl_toolkits.axes_grid1 import make_axes_locatable

### Read the maze

The maze is represented as a connecting graph, which is stored in file.

The maze is a $n \times n$ square.

Total numbers of sites in the maze is $n^2$. 

The nodes of the graph correspond to sites of the maze. 

Two nodes are connected by an edge, if:

1. the corresponding sites are adjacent;
2. there is no wall between them.


In [ ]:
# open the file
data_file = open('./05_data_maze.txt', 'r')

# the first line is the total number of sites of maze
n2,  = [ int(x) for x in data_file.readline().split() ]

# computing square root to get the size of the maze
n = int (math.sqrt(n2))

# size is : n2=nxn
print ('maze size : %d x %d' % (n, n))

# number of neighbours of each site
neib_num = [0] * n2

# list of neighbours of each site
neib_vec = [[]] * n2

# probability to jump to each neighboor
p_vec = [[]] * n2

for idx in range(n2): # each site is described by two lines in the file.
    
    line = data_file.readline()
    # first line tells the number of neighbours (an integer).
    neib_num[idx] = int(line)
    
    line = data_file.readline()
    # second line contains indices of neighbours and jumping probabilities.
    darray = [float (x) for x in line.split()] 
    
    # get indices of neighbours
    neib_vec[idx] = [int (x) for x in darray[::2]]
    
    # get jumping probabilities    
    p_vec[idx] = darray[1::2]
    
data_file.close()

### Draw the maze

In [ ]:
def draw_maze():
    darray = []
    offsets = [[0,1], [0,-1], [-1,0], [1,0]]
    
    for idx in range(n2):
        
        # get x,y from index 
        x = idx % n 
        y = idx // n 
    
        for i in range(4) : # for each direction
            
            xx = x + offsets[i][0]
            yy = y + offsets[i][1]
    
            if (0 <= xx <= n-1) and (0 <= yy <= n-1) :   # if it is a valid site (within the range of maze)
                
                # get index 
                idx1 = yy * n + xx 
                
                if (idx1 not in neib_vec[idx]) and (idx1 > idx) :  # draw a wall, if it is not a neighbour.
                    if xx == x :
                        ymin = min(y, yy) + 0.5 
                        plt.hlines(ymin, x-0.5, x+0.5) # draw a horizontal wall.
                    else : 
                        xmin = min(x, xx) + 0.5 
                        plt.vlines(xmin, y-0.5, y+0.5) # draw a vertical wall.

    # plot four borders                     
    plt.vlines(-0.5, 0.5, n-0.5) 
    plt.vlines(n-0.5, -0.5, n-1.5) 
    plt.hlines(-0.5, -0.5, n-0.5) 
    plt.hlines(n-0.5, -0.5, n-0.5) 
    
    plt.text(-0.3,-0.4, 'A', fontsize=9) 
    plt.text(n-1.3,n-1.3, 'B', fontsize=9) 
    plt.arrow(-1.4,0.0, 0.6, 0, head_width=0.5, head_length=0.2)
    plt.arrow(n-0.2,n-1, 0.6, 0, head_width=0.5, head_length=0.2)

fig = plt.figure(figsize=(9.0,8.5))
   
draw_maze()

### search a path from entrance to exit

The **entrance** corresponds to the **first** node(index $0$).

The **exit** corresponds to the **last** node (index $30^2-1$).

In [ ]:
# Record nodes that have been reached (True or False). 
# Initially, only the entrance is reached. 
reached = [False] * n2
reached[0] = True

# Record the index of the node from which the current node is reached. 
# This list allows us to recover the path. 
parent = [-1] * n2


new_sites = [0]

p = 0
while p < len(new_sites):
    # index of the current node
    idx = new_sites[p]
    for i in range(neib_num[idx]): # loop through its neighbours
        # index of ith neighbour 
        idx_new = neib_vec[idx][i]
        if reached[idx_new] is False: # if this neighbour has not been reached yet.
            
            # add as a newly discovered node.
            new_sites.append(idx_new)
            # set the current node as the parent of the newly discovered node.
            parent[idx_new] = idx
            # mark as reached. 
            reached[idx_new] = True
    p = p + 1
    if reached[n2-1] :  # stop if the exit is reached.
        break

# find the path backwardly using the collected information of parents

# start from the exit
y = n2-1    
path = [y]

while y != 0:   # if not at the entrence
    # go to the parent 
    y = parent[y]
    path.append(y)

# reverse the order     
path = path[::-1]

print (path)

### Visualize the maze and the path

In [ ]:
fig = plt.figure(figsize=(9.0,8.5))
ax = plt.gca() 

draw_maze()

# get x,y of each node along the path
pts = [divmod(x, n) for x in path]

# display the path from entrance to exit
for i in range(0, len(path)-1) :
  if pts[i][1] == pts[i+1][1] :  # if the move is in vertical direction
    plt.vlines(pts[i][1], pts[i][0], pts[i+1][0], linewidth=1.5, linestyle='--',color='k') 
  else : # if the move is in horizontal direction
    plt.hlines(pts[i][0], pts[i][1], pts[i+1][1], linewidth=1.5, linestyle='--', color='k') 
    
fig.patch.set_visible(False)
ax.axis('off')

for tic in ax.yaxis.get_major_ticks():
    tic.tick1On = tic.tick2On = False

for tic in ax.xaxis.get_major_ticks():
    tic.tick1On = tic.tick2On = False

ax.set_xlim( [-2, n+1] )
ax.set_ylim( [-1, n] )
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.show()

### Compute the committor


In [ ]:
## matrix of the linear problem
mat = np.zeros((n2, n2))

# right hand side of the linear problem
b = np.zeros(n2)

# build the linear problem
for idx in range(n2):
    if idx == 0 :   # apply boundary condition q=0 at entrance
        mat[idx][idx] = 1.0    
        b[idx]=0
    if idx == n2-1 :  # apply boundary condition q=1 at exit
        mat[idx][idx] = 1.0
        b[idx]=1
    if idx > 0 and idx < n2 - 1:  # for other sides
        for i in range(neib_num[idx]):  # fill the non-zero entries corresponding to the edges. 
            idx1 = neib_vec[idx][i]
            mat[idx][idx1] = p_vec[idx][i]   # non-diagonal entries are jumping probabilities.
        mat[idx][idx] = -1    # diagonals are one.
        b[idx] = 0     

# compute committor by solving the linear problem
q = np.linalg.solve(mat, b)        

### visualize the computed committor on the maze

In [ ]:

fig = plt.figure(figsize=(9.0,8.5))
ax2 = plt.gca() 

fig.patch.set_visible(False)
ax2.axis('off')

draw_maze()

# display on top of the maze
img=ax2.imshow(q.reshape(n, n), interpolation='none', origin='lower', aspect='auto')

# plot the path 
pts = [divmod(x, n) for x in path[::-1]]
for i in range(0, len(path)-1) :
    if pts[i][1] == pts[i+1][1] :
        plt.vlines(pts[i][1], pts[i][0], pts[i+1][0], linewidth=1.5, linestyle='--',color='k') 
    else : 
        plt.hlines(pts[i][0], pts[i][1], pts[i+1][1], linewidth=1.5, linestyle='--', color='k') 
    
for tic in ax2.yaxis.get_major_ticks():
    tic.tick1On = tic.tick2On = False

for tic in ax2.xaxis.get_major_ticks():
    tic.tick1On = tic.tick2On = False

ax2.set_xlim( [-2, n+1] )
ax2.set_ylim( [-1, n] )
ax2.set_xticklabels([])
ax2.set_yticklabels([])

divider = make_axes_locatable(ax2)
cax = divider.append_axes("right", size="5%", pad=0.05) 
plt.colorbar(img, cax=cax)

plt.tight_layout() 
#cax = plt.colorbar(img, orientation='horizontal')